# 02 · Multi-Armed Bandits
**FOUNDATIONS** · ⏱ 45 min · 🧪 3 exercises · ⭐ 390 XP

*Exploration vs exploitation in the simplest possible RL problem — action values, ε-greedy, optimism, UCB and gradient bandits.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "02-bandits"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Strip reinforcement learning down to its bones: there is **one state**, nothing you do changes
the world, and every action pays out immediately. All that remains is the question that never
goes away — *should I do what has worked so far, or try something I know less about?*

That stripped-down problem is the **k-armed bandit**, named after a row of slot machines
("one-armed bandits"). It is small enough to solve on paper, yet every idea you meet here —
value estimates, incremental updates, ε-greedy exploration, optimism, softmax policies —
reappears, unchanged, in Q-learning, DQN and PPO.

## 01 · The k-armed bandit problem

You face k levers. Pulling lever *a* gives a random reward drawn from a distribution you cannot
see. Each arm has a true mean, its **action value**:

```text
q*(a) = E[ Rₜ | Aₜ = a ]
```

If you knew q* you would always pull `argmax q*(a)` and be done. You do not, so you must
**estimate** it from the rewards you observe, and your estimates are only as good as the data
you choose to collect. Pull an arm rarely and you will know little about it.

Try it yourself. The widget hides the true values; pull arms by hand for a while, then let the
automatic ε-greedy player take over and compare.

In [ ]:
#@title 🎮 **Interactive** · Multi-armed bandit lab
rlcolab.widget("bandit", arms=10, seed=3)

> **ℹ️ NOTE** · **The testbed in this course**
>
> `rlenvs.KArmedBandit(k=10, seed=s)` draws each q*(a) from N(0, 1), and each pull pays
> N(q*(a), 1). That is Sutton & Barto's **10-armed testbed**. `bandit.optimal_action` tells
> *you* (not the agent) which arm is truly best, so you can measure performance.

## 02 · Estimating action values

The natural estimate of q*(a) is the average of the rewards you have received from arm *a*:

```text
Qₜ(a) = (sum of rewards when a was taken) / (number of times a was taken)
```

By the law of large numbers, Qₜ(a) → q*(a) as the arm is pulled infinitely often. Storing every
reward to recompute that mean would be wasteful. Instead, write the mean after *n* rewards in
terms of the mean after *n − 1*:

```text
Qₙ₊₁ = Qₙ + (1/n) [ Rₙ − Qₙ ]
```

Worked example: rewards 1, 2, 3, 6 from one arm, starting at Q₁ = 0.

| n | Rₙ | step 1/n | Qₙ₊₁ = Qₙ + (1/n)(Rₙ − Qₙ) |
|---|---|---|---|
| 1 | 1 | 1 | 0 + 1·(1 − 0) = 1 |
| 2 | 2 | 1/2 | 1 + ½·(2 − 1) = 1.5 |
| 3 | 3 | 1/3 | 1.5 + ⅓·(3 − 1.5) = 2 |
| 4 | 6 | 1/4 | 2 + ¼·(6 − 2) = 3 |

Exactly the mean of (1, 2, 3, 6), using constant memory and one subtraction per step.

> **🔑 KEY IDEA** · **The update rule of reinforcement learning**
>
> **NewEstimate ← OldEstimate + StepSize · [Target − OldEstimate]**
>
> `[Target − OldEstimate]` is an **error**. Each update moves the estimate a fraction of the
> way toward the target. TD learning, Q-learning and even neural-network value losses are all
> this line with a cleverer target.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("02-bandits#0", {"q": "Q = 4 after 3 pulls of an arm. The 4th pull pays 8. What is the new sample-average estimate?", "options": ["6", "5", "4.5", "8"], "answer": 1, "explain": "Q ← 4 + (1/4)(8 − 4) = 5. Check: the old total was 12, the new total 20, and 20 / 4 = 5."})

## 03 · Greedy and ε-greedy action selection

The **greedy** agent always takes `argmax Qₜ(a)`. It exploits, but if an unlucky early reward
makes the best arm look bad, the greedy agent may never pull it again: its estimate is never
corrected, because correcting it requires pulling it.

The fix is to explore a little all the time. An **ε-greedy** agent:

- with probability 1 − ε picks a greedy arm (ties broken at random)
- with probability ε picks an arm uniformly at random, **including** the greedy one

So the greedy arm is chosen with probability (1 − ε) + ε/k, and every other arm with ε/k.
Every arm keeps getting pulled forever, so every estimate converges to the truth.

In [ ]:
#@title 🎮 **Interactive** · ε-greedy distribution
rlcolab.widget("epsilon", values=[0.2, 1.4, 0.9, -0.3, 1.1], epsilon=0.1)

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("02-bandits#1", {"q": "With ε = 0.1 and k = 10 arms, how often does ε-greedy pick the greedy arm (assuming no ties)?", "options": ["90%", "91%", "99%", "10%"], "answer": 1, "explain": "(1 − ε) + ε/k = 0.9 + 0.01 = 0.91. The random branch also lands on the greedy arm 1 time in 10."})

> **⚠️ WARNING** · **Ties matter more than you think**
>
> `np.argmax` returns the **first** maximal index. At the start every Q is 0, so an argmax-greedy
> agent always tries arm 0 first, and in a deterministic task it might never try anything else.
> Break ties at random: `rng.choice(np.flatnonzero(Q == Q.max()))`.

---
### 🧪 Exercise 1 · An ε-greedy bandit agent · ⭐ **120 XP**

Build the three pieces of a complete action-value bandit agent.

1. `incremental_update(old, target, step_size)` — the update rule this whole course runs on: `old + step_size · (target − old)`.
2. `epsilon_greedy(Q, epsilon, rng)` — with probability ε return a uniformly random arm (`rng.integers(len(Q))`); otherwise return a greedy arm, **breaking ties at random** among all arms that share the maximum.
3. `EpsilonGreedyAgent.update(a, r)` — increment `N[a]`, pick the step size (`1 / N[a]` when `alpha is None`, otherwise `alpha`), and move `Q[a]` toward `r`.

`rng` is a NumPy `Generator` (`np.random.default_rng(seed)`), so the agent is reproducible.

**Run** plays 2000 steps on a 10-armed bandit and shows the estimates next to the hidden truth.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 02-bandits/01-epsilon-greedy
import numpy as np
import rlhud
from rlenvs import KArmedBandit


def incremental_update(old, target, step_size):
    """NewEstimate ← OldEstimate + StepSize · [Target − OldEstimate]"""
    # TODO: one line
    raise NotImplementedError


def epsilon_greedy(Q, epsilon, rng):
    """With probability ε pick a uniformly random arm, otherwise a greedy one.

    Ties between equally good greedy arms are broken uniformly at random —
    np.argmax would always pick the lowest index, which silently biases the agent.
    rng is a numpy Generator: use rng.random(), rng.integers(n), rng.choice(arr).
    """
    Q = np.asarray(Q, dtype=float)
    # TODO: explore with probability epsilon, otherwise pick among the argmax arms at random
    raise NotImplementedError


class EpsilonGreedyAgent:
    """Action-value bandit agent.

    alpha=None  -> sample-average estimates (step size 1/N(a))
    alpha=0.1   -> constant step size (recency-weighted average)
    q0          -> initial estimate for every arm (set it high for optimism)
    """

    def __init__(self, k, epsilon=0.1, alpha=None, q0=0.0, seed=None):
        self.k, self.epsilon, self.alpha = k, epsilon, alpha
        self.Q = np.full(k, float(q0))
        self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)

    def act(self):
        return epsilon_greedy(self.Q, self.epsilon, self.rng)

    def update(self, a, r):
        # TODO: count the pull, choose the step size (1/N(a) or alpha),
        #       then move Q[a] toward r with incremental_update
        raise NotImplementedError


if __name__ == "__main__":
    bandit = KArmedBandit(k=10, seed=0)
    agent = EpsilonGreedyAgent(k=10, epsilon=0.1, seed=0)
    rewards, optimal = [], []
    for t in range(2000):
        a = agent.act()
        r = bandit.pull(a)
        agent.update(a, r)
        rewards.append(r)
        optimal.append(100.0 * (a == bandit.optimal_action))

    rlhud.plot(rewards, label="ε = 0.1", title="Reward per step", smooth=100)
    rlhud.plot(optimal, label="ε = 0.1", title="% optimal action", smooth=100)
    rlhud.bars(agent.Q, labels=[f"a{i}" for i in range(10)], title="Estimated Q(a)",
               highlight=int(np.argmax(agent.Q)))
    rlhud.bars(bandit.q_star, labels=[f"a{i}" for i in range(10)], title="True q*(a) (hidden from the agent)",
               highlight=bandit.optimal_action)
    rlhud.stat("Pulls of best arm", int(agent.N[bandit.optimal_action]))
    rlhud.stat("Mean reward", float(np.mean(rewards)))

In [ ]:
#@title ✅ **Check** · An ε-greedy bandit agent (+120 XP) — tests the code you last ran above
rlcolab.check("02-bandits/01-epsilon-greedy")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`incremental_update` is literally one line. In `epsilon_greedy`, draw `rng.random()` once: if it is below ε, explore.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

To break ties: `best = np.flatnonzero(Q == Q.max())` gives the indices of every maximal arm; `rng.choice(best)` picks one. In `update`, increment `self.N[a]` *before* computing `1 / self.N[a]`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`update`: `self.N[a] += 1; step = self.alpha if self.alpha is not None else 1.0 / self.N[a]; self.Q[a] = incremental_update(self.Q[a], r, step)`. Remember to wrap returned arms in `int(...)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("02-bandits/01-epsilon-greedy")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 02-bandits/01-epsilon-greedy
# Reference solution
import numpy as np
import rlhud
from rlenvs import KArmedBandit


def incremental_update(old, target, step_size):
    """NewEstimate ← OldEstimate + StepSize · [Target − OldEstimate]"""
    return old + step_size * (target - old)


def epsilon_greedy(Q, epsilon, rng):
    """With probability ε pick a uniformly random arm, otherwise a greedy one.

    Ties between equally good greedy arms are broken uniformly at random —
    np.argmax would always pick the lowest index, which silently biases the agent.
    """
    Q = np.asarray(Q, dtype=float)
    if rng.random() < epsilon:
        return int(rng.integers(len(Q)))
    best = np.flatnonzero(Q == Q.max())
    return int(rng.choice(best))


class EpsilonGreedyAgent:
    """Action-value bandit agent.

    alpha=None  -> sample-average estimates (step size 1/N(a))
    alpha=0.1   -> constant step size (recency-weighted average)
    q0          -> initial estimate for every arm (set it high for optimism)
    """

    def __init__(self, k, epsilon=0.1, alpha=None, q0=0.0, seed=None):
        self.k, self.epsilon, self.alpha = k, epsilon, alpha
        self.Q = np.full(k, float(q0))
        self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)

    def act(self):
        return epsilon_greedy(self.Q, self.epsilon, self.rng)

    def update(self, a, r):
        self.N[a] += 1
        step = self.alpha if self.alpha is not None else 1.0 / self.N[a]
        self.Q[a] = incremental_update(self.Q[a], r, step)


if __name__ == "__main__":
    bandit = KArmedBandit(k=10, seed=0)
    agent = EpsilonGreedyAgent(k=10, epsilon=0.1, seed=0)
    rewards, optimal = [], []
    for t in range(2000):
        a = agent.act()
        r = bandit.pull(a)
        agent.update(a, r)
        rewards.append(r)
        optimal.append(100.0 * (a == bandit.optimal_action))

    rlhud.plot(rewards, label="ε = 0.1", title="Reward per step", smooth=100)
    rlhud.plot(optimal, label="ε = 0.1", title="% optimal action", smooth=100)
    rlhud.bars(agent.Q, labels=[f"a{i}" for i in range(10)], title="Estimated Q(a)",
               highlight=int(np.argmax(agent.Q)))
    rlhud.bars(bandit.q_star, labels=[f"a{i}" for i in range(10)], title="True q*(a) (hidden from the agent)",
               highlight=bandit.optimal_action)
    rlhud.stat("Pulls of best arm", int(agent.N[bandit.optimal_action]))
    rlhud.stat("Mean reward", float(np.mean(rewards)))

## 04 · Measuring an algorithm: the testbed

A single run on a single bandit is dominated by luck. To compare methods, we average over
**many independent bandit problems** and plot two curves against time:

- **average reward** at step t (what we actually care about)
- **% optimal action** at step t (how often the agent has found the best arm)

Sutton & Barto's classic finding, which you will reproduce: greedy climbs fastest at first,
then flattens at roughly a third optimal; ε = 0.1 explores more, finds the best arm sooner and
ends up near 80% optimal; ε = 0.01 improves slowly but would eventually win, because it wastes
less on random pulls once it knows the answer. You will build this experiment after two more
ideas: constant step sizes and optimism.

## 05 · Nonstationary problems and constant step size

The sample average weights the 1st reward and the 1000th equally. That is right when q* never
changes. In most real problems it does drift (opponents adapt, users change), and old rewards
become misleading. Use a **constant** step size α ∈ (0, 1]:

```text
Qₙ₊₁ = Qₙ + α [ Rₙ − Qₙ ]  =  (1 − α)ⁿ Q₁ + Σᵢ α (1 − α)ⁿ⁻ⁱ Rᵢ
```

This is an **exponential recency-weighted average**: a reward *i* steps old has weight
α(1 − α)ⁱ, which shrinks geometrically. The estimate never fully converges; it keeps
tracking, which is exactly what you want when the target moves.

> **∑ MATH** · **When do step sizes guarantee convergence?**
>
> For a stationary problem, a sequence of step sizes αₙ makes Q converge to q* with
> probability 1 if Σ αₙ = ∞ (steps are big enough to overcome any start) and Σ αₙ² < ∞
> (steps shrink fast enough to average out noise). 1/n satisfies both; a constant α fails the
> second condition, which is why it keeps tracking instead of settling.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("02-bandits#2", {"q": "With constant α = 0.5, how much weight does the most recent reward get in Q?", "options": ["1/n", "0.5", "0.25", "It depends on Q₁"], "answer": 1, "explain": "The newest reward has weight α = 0.5, the one before α(1 − α) = 0.25, then 0.125, … Recent data dominates."})

## 06 · Optimistic initial values

Everything so far starts with Q₁(a) = 0. Start instead with something wildly **optimistic**,
say Q₁(a) = 5 on a testbed where rewards are around 0–2. Now even a purely greedy agent
explores: whichever arm it pulls disappoints it, so its estimate drops below the others and
the agent moves on. Every arm gets tried several times before the estimates settle.

This trick is simple and surprisingly strong on stationary problems (it even beats ε = 0.1 on
the testbed), but it is a *one-off* burst of exploration at the start. If the problem changes
later, optimism has long been used up. Use a constant α with it, otherwise sample averages
wash out the initial value after a single pull.

> **💡 TIP** · **Optimism reappears later**
>
> Initialising Q-values optimistically is still a standard trick for tabular Q-learning, and
> "optimism in the face of uncertainty" is the idea behind UCB below.

Now put the testbed together and watch greedy, ε-greedy, optimistic-greedy, and the
sample-average vs constant-α race on a drifting bandit, all in one experiment.

---
### 🧪 Exercise 2 · The 10-armed testbed · ⭐ **150 XP**

One bandit run is noise. To compare algorithms honestly, Sutton & Barto average **hundreds of independent bandit problems**. Build that experiment.

1. `run_testbed(make_agent, n_runs, n_steps, k, seed, nonstationary)` — for run `i`, create `KArmedBandit(k=k, seed=seed + i, nonstationary=nonstationary)` and `agent = make_agent(seed + i)`, then play `n_steps` of `act → pull → update`. Return `(avg_reward, pct_optimal)`, two arrays of length `n_steps` averaged over runs (`pct_optimal` in **percent**, measured against `bandit.optimal_action` before the pull).
2. `compare_epsilons(epsilons, n_runs, n_steps, seed)` — return `{eps: run_testbed(...)}` using `EpsilonGreedyAgent(10, epsilon=eps, seed=s)` (already implemented above).

The check confirms the classic result: ε = 0.1 beats pure greedy. **Run** plots greedy vs ε = 0.01 vs ε = 0.1, optimistic initial values, and a nonstationary race between sample averages and constant α.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 02-bandits/02-testbed
import numpy as np
import rlhud
from rlenvs import KArmedBandit


class EpsilonGreedyAgent:
    """The agent from exercise 1, complete. alpha=None → sample averages."""

    def __init__(self, k, epsilon=0.1, alpha=None, q0=0.0, seed=None):
        self.k, self.epsilon, self.alpha = k, epsilon, alpha
        self.Q = np.full(k, float(q0))
        self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)

    def act(self):
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.k))
        return int(self.rng.choice(np.flatnonzero(self.Q == self.Q.max())))

    def update(self, a, r):
        self.N[a] += 1
        step = self.alpha if self.alpha is not None else 1.0 / self.N[a]
        self.Q[a] += step * (r - self.Q[a])


def run_testbed(make_agent, n_runs=200, n_steps=1000, k=10, seed=0, nonstationary=False):
    """Average an agent's learning curve over many independent bandit problems.

    For run i (0 … n_runs-1):
        bandit = KArmedBandit(k=k, seed=seed + i, nonstationary=nonstationary)
        agent  = make_agent(seed + i)
    then play n_steps:  a = agent.act(); r = bandit.pull(a); agent.update(a, r)

    Returns (avg_reward, pct_optimal): NumPy arrays of length n_steps.
        avg_reward[t]  = mean over runs of the reward received at step t
        pct_optimal[t] = 100 × fraction of runs whose action at step t was
                         bandit.optimal_action (checked *before* the pull)
    """
    rewards = np.zeros((n_runs, n_steps))
    optimal = np.zeros((n_runs, n_steps))
    # TODO: for each run, build the bandit and the agent exactly as described above,
    #       play n_steps, and fill rewards[i, t] and optimal[i, t]
    raise NotImplementedError
    return rewards.mean(axis=0), 100.0 * optimal.mean(axis=0)


def compare_epsilons(epsilons, n_runs=200, n_steps=1000, seed=0):
    """Run the testbed once per ε (sample-average agents, q0 = 0).

    Returns {epsilon: (avg_reward, pct_optimal)}.
    """
    results = {}
    for eps in epsilons:
        # TODO: results[eps] = run_testbed(...) with a make_agent that builds
        #       EpsilonGreedyAgent(10, epsilon=eps, seed=s)
        raise NotImplementedError
    return results


if __name__ == "__main__":
    runs = 100
    res = compare_epsilons([0.0, 0.01, 0.1], n_runs=runs, n_steps=1000)
    rows = []
    for eps, (rew, opt) in res.items():
        name = "greedy" if eps == 0 else f"ε = {eps}"
        rlhud.plot(rew, label=name, title=f"Average reward · {runs} runs", smooth=10)
        rlhud.plot(opt, label=name, title="% optimal action", smooth=10)
        rows.append([name, round(float(rew[-500:].mean()), 3), round(float(opt[-500:].mean()), 1)])

    # Optimism as an exploration strategy: greedy, but every arm starts at Q₀ = 5
    rew, opt = run_testbed(lambda s: EpsilonGreedyAgent(10, epsilon=0.0, alpha=0.1, q0=5.0, seed=s),
                           n_runs=runs, n_steps=1000)
    rlhud.plot(opt, label="optimistic greedy (Q₀=5, α=0.1)", title="% optimal action", smooth=10)
    rows.append(["optimistic Q₀=5", round(float(rew[-500:].mean()), 3), round(float(opt[-500:].mean()), 1)])
    rlhud.table(rows, headers=["agent", "reward (last 500)", "% optimal (last 500)"], title="Testbed summary")

    # Nonstationary: arms drift. Constant α keeps tracking; sample averages freeze.
    for alpha, name in [(None, "sample average"), (0.1, "constant α = 0.1")]:
        rew, opt = run_testbed(lambda s, a=alpha: EpsilonGreedyAgent(10, epsilon=0.1, alpha=a, seed=s),
                               n_runs=30, n_steps=3000, nonstationary=True)
        rlhud.plot(opt, label=name, title="Nonstationary bandit · % optimal", smooth=50)

In [ ]:
#@title ✅ **Check** · The 10-armed testbed (+150 XP) — tests the code you last ran above
rlcolab.check("02-bandits/02-testbed")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Allocate `rewards = np.zeros((n_runs, n_steps))` and the same for `optimal`; one row per run. At the end, `.mean(axis=0)` averages over runs.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Inside the run loop: `a = agent.act(); optimal[i, t] = a == bandit.optimal_action; r = bandit.pull(a); agent.update(a, r); rewards[i, t] = r`. Record optimality *before* pulling — a nonstationary bandit changes its best arm after each pull.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`compare_epsilons`: `results[eps] = run_testbed(lambda s, e=eps: EpsilonGreedyAgent(10, epsilon=e, seed=s), n_runs=n_runs, n_steps=n_steps, seed=seed)`. The `e=eps` default argument freezes the loop variable inside the lambda.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("02-bandits/02-testbed")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 02-bandits/02-testbed
# Reference solution
import numpy as np
import rlhud
from rlenvs import KArmedBandit


class EpsilonGreedyAgent:
    """The agent from exercise 1, complete. alpha=None → sample averages."""

    def __init__(self, k, epsilon=0.1, alpha=None, q0=0.0, seed=None):
        self.k, self.epsilon, self.alpha = k, epsilon, alpha
        self.Q = np.full(k, float(q0))
        self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)

    def act(self):
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.k))
        return int(self.rng.choice(np.flatnonzero(self.Q == self.Q.max())))

    def update(self, a, r):
        self.N[a] += 1
        step = self.alpha if self.alpha is not None else 1.0 / self.N[a]
        self.Q[a] += step * (r - self.Q[a])


def run_testbed(make_agent, n_runs=200, n_steps=1000, k=10, seed=0, nonstationary=False):
    """Average an agent's learning curve over many independent bandit problems.

    For run i (0 … n_runs-1):
        bandit = KArmedBandit(k=k, seed=seed + i, nonstationary=nonstationary)
        agent  = make_agent(seed + i)
    then play n_steps:  a = agent.act(); r = bandit.pull(a); agent.update(a, r)

    Returns (avg_reward, pct_optimal): NumPy arrays of length n_steps.
        avg_reward[t]  = mean over runs of the reward received at step t
        pct_optimal[t] = 100 × fraction of runs whose action at step t was
                         bandit.optimal_action (checked *before* the pull)
    """
    rewards = np.zeros((n_runs, n_steps))
    optimal = np.zeros((n_runs, n_steps))
    for i in range(n_runs):
        bandit = KArmedBandit(k=k, seed=seed + i, nonstationary=nonstationary)
        agent = make_agent(seed + i)
        for t in range(n_steps):
            a = agent.act()
            optimal[i, t] = a == bandit.optimal_action
            r = bandit.pull(a)
            agent.update(a, r)
            rewards[i, t] = r
    return rewards.mean(axis=0), 100.0 * optimal.mean(axis=0)


def compare_epsilons(epsilons, n_runs=200, n_steps=1000, seed=0):
    """Run the testbed once per ε (sample-average agents, q0 = 0).

    Returns {epsilon: (avg_reward, pct_optimal)}.
    """
    results = {}
    for eps in epsilons:
        results[eps] = run_testbed(lambda s, e=eps: EpsilonGreedyAgent(10, epsilon=e, seed=s),
                                   n_runs=n_runs, n_steps=n_steps, seed=seed)
    return results


if __name__ == "__main__":
    runs = 100
    res = compare_epsilons([0.0, 0.01, 0.1], n_runs=runs, n_steps=1000)
    rows = []
    for eps, (rew, opt) in res.items():
        name = "greedy" if eps == 0 else f"ε = {eps}"
        rlhud.plot(rew, label=name, title=f"Average reward · {runs} runs", smooth=10)
        rlhud.plot(opt, label=name, title="% optimal action", smooth=10)
        rows.append([name, round(float(rew[-500:].mean()), 3), round(float(opt[-500:].mean()), 1)])

    # Optimism as an exploration strategy: greedy, but every arm starts at Q₀ = 5
    rew, opt = run_testbed(lambda s: EpsilonGreedyAgent(10, epsilon=0.0, alpha=0.1, q0=5.0, seed=s),
                           n_runs=runs, n_steps=1000)
    rlhud.plot(opt, label="optimistic greedy (Q₀=5, α=0.1)", title="% optimal action", smooth=10)
    rows.append(["optimistic Q₀=5", round(float(rew[-500:].mean()), 3), round(float(opt[-500:].mean()), 1)])
    rlhud.table(rows, headers=["agent", "reward (last 500)", "% optimal (last 500)"], title="Testbed summary")

    # Nonstationary: arms drift. Constant α keeps tracking; sample averages freeze.
    for alpha, name in [(None, "sample average"), (0.1, "constant α = 0.1")]:
        rew, opt = run_testbed(lambda s, a=alpha: EpsilonGreedyAgent(10, epsilon=0.1, alpha=a, seed=s),
                               n_runs=30, n_steps=3000, nonstationary=True)
        rlhud.plot(opt, label=name, title="Nonstationary bandit · % optimal", smooth=50)

## 07 · Upper confidence bound (UCB) action selection

ε-greedy explores *blindly*: a random pull is equally likely to go to an arm you have tried
500 times as to one you have tried twice. It would be smarter to prefer arms that are both
promising **and** uncertain. UCB picks

```text
Aₜ = argmaxₐ [ Qₜ(a) + c · √( ln t / Nₜ(a) ) ]
```

where Nₜ(a) is how many times *a* has been pulled. The square-root term is an **uncertainty
bonus**:

- each pull of *a* raises Nₜ(a) and shrinks its bonus
- time passing raises ln t and slowly grows the bonus of every arm that is *not* pulled
- arms with Nₜ(a) = 0 count as maximally uncertain and are tried first
- c > 0 controls how bold the exploration is (c = 2 is the textbook setting)

Worked example, c = 1, t = 11: arm A has Q = 1.0 after 10 pulls, arm B has Q = 0.5 after 1 pull.
A scores 1.0 + √(2.40/10) ≈ 1.49, B scores 0.5 + √(2.40/1) ≈ 2.05. UCB pulls B: one sample is
not enough to rule it out.

In [ ]:
#@title 🧠 **Knowledge check 4** · +25 XP — run, then pick an answer
rlcolab.quiz("02-bandits#3", {"q": "Arm a has never been pulled while every other arm has been pulled 50 times. What does UCB do?", "options": ["Ignores it until ε fires", "Pulls it — its bonus is infinite", "Pulls it only if its Q is highest", "Pulls it with probability 1/k"], "answer": 1, "explain": "With N(a) = 0 the bound is treated as +∞, so every arm is tried once before any estimate is trusted."})

---
### 🧪 Exercise 3 · Upper Confidence Bound · ⭐ **120 XP**

Implement UCB action selection, then the loop that uses it.

1. `ucb_scores(Q, N, t, c)` — return an array with `Q[a] + c · sqrt(ln t / N[a])` for every arm, and `+inf` for arms with `N[a] == 0`.
2. `ucb_action(Q, N, t, c)` — the arm with the highest score (`np.argmax`, ties → lowest index).
3. `run_ucb(bandit, steps, c)` — for `t = 1 … steps`: choose with UCB, `r = bandit.pull(a)`, update the count and the **sample-average** estimate, and record `r` and whether `a == bandit.optimal_action`.

A working ε-greedy baseline `run_eps_greedy` is provided. **Run** compares both over 100 seeded bandits.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 02-bandits/03-ucb
import numpy as np
import rlhud
from rlenvs import KArmedBandit


def ucb_scores(Q, N, t, c):
    """Upper confidence bound of every arm:  Q(a) + c · sqrt( ln t / N(a) ).

    Arms that were never tried (N = 0) get +inf: they are maximally uncertain.
    t is the number of the current step, counting from 1.
    """
    Q = np.asarray(Q, dtype=float)
    N = np.asarray(N, dtype=float)
    # TODO: start from an array of +inf, fill in the bound for the arms with N > 0
    raise NotImplementedError


def ucb_action(Q, N, t, c):
    """The arm with the highest upper bound (ties → lowest index, like np.argmax)."""
    # TODO
    raise NotImplementedError


def run_ucb(bandit, steps, c=2.0):
    """Play `steps` pulls with UCB and sample-average estimates.

    Returns (rewards, optimal) — two lists of length `steps`, where optimal[i]
    is True if the arm chosen at step i was the bandit's optimal arm.
    """
    k = bandit.k
    Q, N = np.zeros(k), np.zeros(k)
    rewards, optimal = [], []
    for t in range(1, steps + 1):
        # TODO: choose with ucb_action, pull, update N and the sample average Q
        raise NotImplementedError
    return rewards, optimal


def run_eps_greedy(bandit, steps, eps=0.1, seed=0):
    """Baseline for comparison (already written for you)."""
    rng = np.random.default_rng(seed)
    k = bandit.k
    Q, N = np.zeros(k), np.zeros(k)
    rewards, optimal = [], []
    for t in range(steps):
        a = int(rng.integers(k)) if rng.random() < eps else int(rng.choice(np.flatnonzero(Q == Q.max())))
        r = bandit.pull(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards.append(r)
        optimal.append(a == bandit.optimal_action)
    return rewards, optimal


if __name__ == "__main__":
    runs, steps = 100, 1000
    curves = {"UCB c=2": np.zeros(steps), "ε-greedy 0.1": np.zeros(steps)}
    for i in range(runs):
        r, _ = run_ucb(KArmedBandit(k=10, seed=i), steps, c=2.0)
        curves["UCB c=2"] += np.array(r) / runs
        r, _ = run_eps_greedy(KArmedBandit(k=10, seed=i), steps, eps=0.1, seed=i)
        curves["ε-greedy 0.1"] += np.array(r) / runs
        rlhud.progress(i, runs, "Testbed runs")
    for name, y in curves.items():
        rlhud.plot(y, label=name, title=f"Average reward ({runs} runs)", smooth=10)
        rlhud.stat(f"{name} · mean reward", float(y.mean()))

    # What UCB "thinks" halfway through one run
    bandit = KArmedBandit(k=10, seed=0)
    Q, N = np.zeros(10), np.zeros(10)
    for t in range(1, 201):
        a = ucb_action(Q, N, t, 2.0)
        r = bandit.pull(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
    rlhud.bars(ucb_scores(Q, N, 200, 2.0), labels=[f"a{i}" for i in range(10)],
               title="UCB scores after 200 steps", highlight=bandit.optimal_action)
    rlhud.bars(N, labels=[f"a{i}" for i in range(10)], title="Pull counts after 200 steps",
               highlight=bandit.optimal_action)

In [ ]:
#@title ✅ **Check** · Upper Confidence Bound (+120 XP) — tests the code you last ran above
rlcolab.check("02-bandits/03-ucb")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Use `np.log` (natural log). Start with `scores = np.full(len(Q), np.inf)` and only compute the bound where `N > 0`, so you never divide by zero.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

A boolean mask does it in one line: `tried = N > 0; scores[tried] = Q[tried] + c * np.sqrt(np.log(t) / N[tried])`. In the loop, `t` already starts at 1 — don't pass `t - 1` or ln 0 appears.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Loop body: `a = ucb_action(Q, N, t, c); r = bandit.pull(a); N[a] += 1; Q[a] += (r - Q[a]) / N[a]; rewards.append(r); optimal.append(a == bandit.optimal_action)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("02-bandits/03-ucb")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 02-bandits/03-ucb
# Reference solution
import numpy as np
import rlhud
from rlenvs import KArmedBandit


def ucb_scores(Q, N, t, c):
    """Upper confidence bound of every arm:  Q(a) + c · sqrt( ln t / N(a) ).

    Arms that were never tried (N = 0) get +inf: they are maximally uncertain.
    t is the number of the current step, counting from 1.
    """
    Q = np.asarray(Q, dtype=float)
    N = np.asarray(N, dtype=float)
    scores = np.full(len(Q), np.inf)
    tried = N > 0
    scores[tried] = Q[tried] + c * np.sqrt(np.log(t) / N[tried])
    return scores


def ucb_action(Q, N, t, c):
    """The arm with the highest upper bound (ties → lowest index, like np.argmax)."""
    return int(np.argmax(ucb_scores(Q, N, t, c)))


def run_ucb(bandit, steps, c=2.0):
    """Play `steps` pulls with UCB and sample-average estimates.

    Returns (rewards, optimal) — two lists of length `steps`, where optimal[i]
    is True if the arm chosen at step i was the bandit's optimal arm.
    """
    k = bandit.k
    Q, N = np.zeros(k), np.zeros(k)
    rewards, optimal = [], []
    for t in range(1, steps + 1):
        a = ucb_action(Q, N, t, c)
        r = bandit.pull(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards.append(r)
        optimal.append(a == bandit.optimal_action)
    return rewards, optimal


def run_eps_greedy(bandit, steps, eps=0.1, seed=0):
    """Baseline for comparison (already written for you)."""
    rng = np.random.default_rng(seed)
    k = bandit.k
    Q, N = np.zeros(k), np.zeros(k)
    rewards, optimal = [], []
    for t in range(steps):
        a = int(rng.integers(k)) if rng.random() < eps else int(rng.choice(np.flatnonzero(Q == Q.max())))
        r = bandit.pull(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards.append(r)
        optimal.append(a == bandit.optimal_action)
    return rewards, optimal


if __name__ == "__main__":
    runs, steps = 100, 1000
    curves = {"UCB c=2": np.zeros(steps), "ε-greedy 0.1": np.zeros(steps)}
    for i in range(runs):
        r, _ = run_ucb(KArmedBandit(k=10, seed=i), steps, c=2.0)
        curves["UCB c=2"] += np.array(r) / runs
        r, _ = run_eps_greedy(KArmedBandit(k=10, seed=i), steps, eps=0.1, seed=i)
        curves["ε-greedy 0.1"] += np.array(r) / runs
        rlhud.progress(i, runs, "Testbed runs")
    for name, y in curves.items():
        rlhud.plot(y, label=name, title=f"Average reward ({runs} runs)", smooth=10)
        rlhud.stat(f"{name} · mean reward", float(y.mean()))

    # What UCB "thinks" halfway through one run
    bandit = KArmedBandit(k=10, seed=0)
    Q, N = np.zeros(10), np.zeros(10)
    for t in range(1, 201):
        a = ucb_action(Q, N, t, 2.0)
        r = bandit.pull(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
    rlhud.bars(ucb_scores(Q, N, 200, 2.0), labels=[f"a{i}" for i in range(10)],
               title="UCB scores after 200 steps", highlight=bandit.optimal_action)
    rlhud.bars(N, labels=[f"a{i}" for i in range(10)], title="Pull counts after 200 steps",
               highlight=bandit.optimal_action)

> **ℹ️ NOTE** · **Why UCB rarely leaves the bandit chapter**
>
> UCB needs counts N(a) for every action, and the ln t schedule assumes a stationary problem.
> With large state spaces or function approximation those counts do not exist, so later
> chapters mostly use ε-greedy or stochastic policies, though "count-based exploration
> bonuses" are a live idea in deep RL.

## 08 · Gradient bandits: learning preferences instead of values

A different approach skips value estimates entirely. Keep a numerical **preference** Hₜ(a)
for each arm and act stochastically with the **softmax** (Gibbs/Boltzmann) distribution:

```text
πₜ(a) = exp(Hₜ(a)) / Σ_b exp(Hₜ(b))
```

Only *differences* between preferences matter: adding 100 to every H leaves π unchanged. Play
with the temperature slider: high temperature flattens the distribution, low temperature makes
it nearly greedy.

In [ ]:
#@title 🎮 **Interactive** · Softmax policy
rlcolab.widget("softmax", prefs=[1.0, 2.0, 0.5, 0.0], temperature=1.0)

After taking Aₜ and receiving Rₜ, nudge the preferences with stochastic gradient ascent on
expected reward:

```text
Hₜ₊₁(Aₜ) = Hₜ(Aₜ) + α (Rₜ − R̄ₜ)(1 − πₜ(Aₜ))
Hₜ₊₁(a)  = Hₜ(a)  − α (Rₜ − R̄ₜ) πₜ(a)         for a ≠ Aₜ
```

R̄ₜ is the average reward so far, the **baseline**. Better than average → make the chosen arm
more likely (and the others less); worse → the reverse. In code it is a few lines:

```python
pi = np.exp(H - H.max()); pi /= pi.sum()      # subtract max for numerical safety
a = rng.choice(k, p=pi)
r = bandit.pull(a)
baseline += (r - baseline) / t
one_hot = np.eye(k)[a]
H += alpha * (r - baseline) * (one_hot - pi)
```

> **🔑 KEY IDEA** · **You have just seen your first policy gradient**
>
> "Increase the log-probability of actions in proportion to how much better than a baseline
> they turned out" is precisely REINFORCE with a baseline, which leads to actor-critic and PPO.
> The baseline does not change the expected update, but without it the variance is much larger.

In [ ]:
#@title 🧠 **Knowledge check 5** · +25 XP — run, then pick an answer
rlcolab.quiz("02-bandits#4", {"q": "In a gradient bandit, the chosen arm paid exactly the baseline (Rₜ = R̄ₜ). What happens to the preferences?", "options": ["The chosen arm's preference rises", "Nothing changes", "All preferences reset to 0", "The chosen arm's preference falls"], "answer": 1, "explain": "Every update is multiplied by (Rₜ − R̄ₜ) = 0. Only better- or worse-than-average outcomes carry information."})

## 09 · Pitfalls

- **argmax without random tie-breaking** biases early behaviour toward low indices.
- **Off-by-one in 1/N**: increment N(a) *before* computing the step size, otherwise the first
  update uses a step of 1/0.
- **Optimism with sample averages** is wiped out after one pull: the first reward replaces Q₁.
- **ln 0**: start UCB's time counter at t = 1.
- **One run proves nothing**: compare algorithms over hundreds of seeded runs, as in the
  testbed exercise, before believing a curve.

## 10 · What carries forward

| Bandit idea | Where it returns |
|---|---|
| Incremental update toward a target | Monte Carlo, TD(0), SARSA, Q-learning, DQN loss |
| ε-greedy | Every tabular control method, DQN |
| Constant α for nonstationarity | Policy changes make value targets nonstationary in all of RL |
| Optimistic initial values | Tabular Q-learning, exploration bonuses |
| Softmax preferences + baseline | REINFORCE, actor-critic, PPO |

Bandits ignore one thing: actions that change **which situation you are in next**. The next
lesson adds states, and with them the full reinforcement learning problem: the Markov decision
process.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("02-bandits")